In [9]:
%matplotlib tk
import torch
import os
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots
import matplotlib.animation as animation

plt.style.use(['science','notebook', 'grid'])
device = torch.device('cuda' if torch.cuda.is_available( ) else 'cpu')

In [101]:
class QuadTreeNode:
    def __init__(self, x_min, x_max, y_min, y_max, indices, positions, masses, max_particles=3):
        self.x_min, self.x_max = x_min, x_max
        self.y_min, self.y_max = y_min, y_max
        self.indices = indices
        self.children = []
        self.center_of_mass = None
        self.total_mass = None
        self.is_leaf = len(indices) <= max_particles
        if len(indices) == 0:
            self.center_of_mass = torch.zeros(2, device=positions.device)
            self.total_mass = 0.0
        else:
            pos = positions[:, indices]
            m = masses[indices]
            total_mass = m.sum()
            if total_mass > 0:
                com = (pos * m).sum(dim=1) / total_mass
            else:
                com = torch.zeros(2, device=positions.device)
            self.center_of_mass = com
            self.total_mass = total_mass
        if not self.is_leaf:
            self.subdivide(positions, masses, max_particles)

    def subdivide(self, positions, masses, max_particles):
        xm, xM, ym, yM = self.x_min, self.x_max, self.y_min, self.y_max
        xc, yc = (xm + xM) / 2, (ym + yM) / 2
        quads = [
            (xm, xc, ym, yc),  # SW
            (xc, xM, ym, yc),  # SE
            (xm, xc, yc, yM),  # NW
            (xc, xM, yc, yM),  # NE
        ]
        pos = positions[:, self.indices]
        for (qx0, qx1, qy0, qy1) in quads:
            mask = (
                (pos[0] >= qx0) & (pos[0] < qx1) &
                (pos[1] >= qy0) & (pos[1] < qy1)
            )
            child_indices = [self.indices[i] for i, m in enumerate(mask) if m]
            self.children.append(
                QuadTreeNode(qx0, qx1, qy0, qy1, child_indices, positions, masses, max_particles)
            )

def barnes_hut_accel(positions, masses, theta=10, G=1.0, softening=1e-3, max_particles=1):
    N = positions.shape[1]
    x_min, x_max = positions[0].min().item(), positions[0].max().item()
    y_min, y_max = positions[1].min().item(), positions[1].max().item()
    indices = list(range(N))
    root = QuadTreeNode(x_min, x_max, y_min, y_max, indices, positions, masses, max_particles)
    acc = torch.zeros_like(positions)
    def compute_force(i, node):
        if node.total_mass == 0 or (node.is_leaf and i in node.indices and len(node.indices) == 1):
            return torch.zeros(2, device=positions.device)
        dx = node.center_of_mass[0] - positions[0, i]
        dy = node.center_of_mass[1] - positions[1, i]
        dist2 = dx**2 + dy**2 + softening**2
        dist = torch.sqrt(dist2)
        width = node.x_max - node.x_min
        if node.is_leaf or width / dist < theta:
            #if (dist < 0.1): print(len(node.indices), dist.item())
            if dist > 0:
                force = G * node.total_mass * torch.tensor([dx, dy], device=positions.device) / (dist2 * dist)
                return force
            else:
                return torch.zeros(2, device=positions.device)
        else:
            force = torch.zeros(2, device=positions.device)
            for child in node.children:
                force += compute_force(i, child)
            return force
    for i in range(N):
        acc[:, i] = compute_force(i, root)
    return acc


In [102]:
def gravity_accel(r, gc, N):
    ones = torch.ones(N).to(device)
    ax = gc * (r[0] - ones) / ((torch.sqrt((r[0]-ones)**2 + (r[1]-ones)**2))**3+1e-6)
    ay = gc * (r[1] - ones) / ((torch.sqrt((r[0]-ones)**2 + (r[1]-ones)**2))**3+1e-6)
    return ax, ay

# Adaptive time stepping version of motion

def motion_adaptive(r, v, masses, ts, dt_init, d_cutoff, N, dt_min=1e-8, dt_max=1e-3, safety=20):
    rs = torch.zeros((ts, r.shape[0], r.shape[1]), device=r.device)
    vs = torch.zeros((ts, v.shape[0], v.shape[1]), device=v.device)
    dts = torch.zeros(ts)
    rs[0] = r
    vs[0] = v
    dt = dt_init
    dts[0] = dt
    for i in range(1, ts):
        print(f"Percent Complete: {(i+1) * 100 / ts:0.2f}%", end='\r')
        # Estimate max velocity for adaptive dt
        vmag = v.norm(dim=0)
        vmax = vmag.max().item()
        if vmax > 0:
            dt = min(max(safety * d_cutoff / vmax, dt_min), dt_max)
        else:
            dt = dt_max
        dts[i] = dt
        # Update positions and velocities
        ax, ay = gravity_accel(r, 0, N)
        acc = barnes_hut_accel(r, masses, theta=2.5, G=0.5e5)
        ax = ax - acc[0]
        ay = ay - acc[1]
        r = r + v*dt
        r[0, :] = r[0, :] - ax*dt**2/2
        v[0, :] = v[0, :] - ax*dt
        r[1, :] = r[1, :] - ay*dt**2/2
        v[1, :] = v[1, :] - ay*dt
        rs[i] = r
        vs[i] = v
    return rs, vs, dts

In [105]:
# Use adaptive time stepping for the main simulation
N = 300
dt_init = 3.3e-5
t_steps = 30
v0 = 450
L = 1
mass = 0.85e-2
r = 1.5 * torch.rand((2,N)).to(device) + 0.25
ixr = r[0]>1
ixl = r[0]<=1 
ids = torch.arange(N)
ids_pairs = torch.combinations(ids,2).to(device)
v = torch.zeros((2,N)).to(device)
v[0] = -v0*(r[1]-1)
v[1] = v0*(r[0]-1)
masses = mass*torch.ones(N).to(device)
radius = 0.0005
rs, vs, dts = motion_adaptive(r, v, masses, ts=t_steps, dt_init=dt_init, d_cutoff=2*radius, N=N)

KeyboardInterrupt: 

In [69]:
# Optional: plot adaptive time step history
plt.figure()
plt.plot(dts.cpu().numpy())
plt.xlabel('Step')
plt.ylabel('dt')
plt.title('Adaptive time step history')
plt.show()
print(dts[1])

tensor(4.2725e-05)


In [104]:
fig, ax = plt.subplots(1,1,figsize=(5,5))
ax.clear()
vmin = 0
vmax = 2.5
ax.set_xlim(0,vmax)
ax.set_ylim(0,vmax)
markersize = 2*2 * radius * ax.get_window_extent().width  / (vmax-vmin) * 72./fig.dpi
red, = ax.plot([], [], 'o', color='red', markersize=markersize)
blue, = ax.plot([], [], 'o', color='blue', markersize=markersize)

# Only transfer to CPU for plotting, keep all other operations on GPU
def animate(i):
    # rs is already on GPU; only transfer the minimal data needed for plotting
    xred = rs[i][0][ixr].detach().cpu().numpy()
    yred = rs[i][1][ixr].detach().cpu().numpy()
    xblue = rs[i][0][ixl].detach().cpu().numpy()
    yblue = rs[i][1][ixl].detach().cpu().numpy()
    red.set_data(xred, yred)
    blue.set_data(xblue, yblue)
    return red, blue

writer = animation.FFMpegWriter(fps=30)
ani = animation.FuncAnimation(fig, animate, frames=t_steps, interval=50, blit=True)
#ani.save(filename="/Users/hasan/Python Animations/Self Gas Gravity.gif", writer="pillow")